<a href="https://colab.research.google.com/github/ArkanUbaidillah/BigData26_B_2411537001_ArkanUbaidillahWarman/blob/main/Praktikum3/BD_P03_2411537001_ArkanUbaidillahWarman_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum Big Data 3
## Pengumpulan dan Pra-pemrosesan Data untuk Big Data




## K-1. Menyiapkan Lingkungan dan Memindahkan Perkakas dari Praktikum 1

Struktur `catatan` dipakai untuk biaya komputasi (waktu dan memori).
Struktur `lineage` dipakai untuk asal-usul data.
Keduanya disimpan sebagai artefak pada K-10.

**Keputusan folder:** `DIR_SIMPAN` diarahkan ke `MyDrive/BigData/Praktikum3` agar artefak praktikum ini tidak menimpa hasil Praktikum 2.


In [1]:
import sys, time, os, json, sqlite3, gc
import pandas as pd, numpy as np, psutil

for nama in ["pandas", "numpy", "pyarrow", "requests", "sklearn", "sqlalchemy"]:
    try:
        mod = __import__(nama)
        print(f"{nama:11s}: {mod.__version__}")
    except ImportError:
        print(f"{nama:11s}: BELUM TERPASANG")

from google.colab import drive
drive.mount("/content/drive")

DIR_MENTAH = "/content/lapisan_mentah"
DIR_KURASI = "/content/lapisan_terkurasi"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum3"
for d in (DIR_MENTAH, DIR_KURASI, DIR_SIMPAN):
    os.makedirs(d, exist_ok=True)

# --- Dipakai kembali dari Praktikum 1 / K-3 ---
proses = psutil.Process(os.getpid())
catatan = []
def rss_mb():
    return proses.memory_info().rss / 1024**2
def ukur(label, fungsi):
    m0, t0 = rss_mb(), time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 2),
                     "delta_rss_mb": round(rss_mb() - m0, 1)})
    print(f"[{label}] {detik:.2f} s")
    return hasil

# --- Baru di Praktikum 3: catatan asal-usul data (lineage) ---
lineage = []
def catat_sumber(nama, asal, jumlah_baris, keterangan=""):
    lineage.append({
        "sumber": nama, "asal": asal,
        "diambil_pada": pd.Timestamp.now("UTC").isoformat(),
        "jumlah_baris": jumlah_baris, "keterangan": keterangan,
    })
    print(f"[lineage] {nama}: {jumlah_baris:,} baris")


pandas     : 2.2.3
numpy      : 2.1.3
pyarrow    : 23.0.1
requests   : 2.32.4
sklearn    : 1.6.1
sqlalchemy : 2.0.52
Mounted at /content/drive


## K-2. Akuisisi 1 — Unduhan Berkala yang Tahan Gagal

Unduhan ditulis ke berkas `.part` dulu, baru diganti nama secara atomik dengan `os.replace`.
Jika berkas final sudah ada dan ukurannya > 0, unduhan dilewati (cache).
Retry memakai exponential backoff.

**Keputusan kolom:** daftar `KOLOM` menyertakan `PULocationID` dan `DOLocationID` sebagai kunci join pada K-8.

**Perbaikan teknis:** unduhan memakai `iter_content` (bukan `r.raw`) agar CSV yang dikirim gzip otomatis didekompresi sebelum disimpan. Cache yang masih berformat gzip akan diunduh ulang.


In [2]:
import requests, shutil, gzip

def unduh_aman(url, tujuan, percobaan=3, jeda_awal=2):
    """Unduh dengan retry + penulisan atomik. Melewati unduhan bila file sudah ada.

    Catatan: jangan tulis r.raw langsung. Server TLC mengirim CSV dengan
    Content-Encoding gzip; r.raw masih terkompresi dan membuat pd.read_csv gagal.
    Pakai iter_content agar requests mendekompresi otomatis.
    """
    def _berkas_ok(path):
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        # tolak sisa unduhan gzip yang belum didekompresi
        with open(path, "rb") as fh:
            kepala = fh.read(2)
        if kepala == b"\x1f\x8b":
            print("cache rusak (masih gzip), unduh ulang:", os.path.basename(path))
            try:
                os.remove(path)
            except OSError:
                pass
            return False
        return True

    if _berkas_ok(tujuan):
        print("cache ditemukan:", os.path.basename(tujuan))
        return tujuan

    sementara = tujuan + ".part"
    for i in range(percobaan):
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                # iter_content = body sudah didekompresi oleh requests
                with open(sementara, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                if os.path.getsize(sementara) == 0:
                    raise IOError("berkas kosong")
                # jaga-jaga bila server mengirim gzip mentah tanpa header encoding
                with open(sementara, "rb") as fh:
                    mag = fh.read(2)
                if mag == b"\x1f\x8b":
                    with gzip.open(sementara, "rb") as gz, open(sementara + ".dec", "wb") as out:
                        shutil.copyfileobj(gz, out)
                    os.replace(sementara + ".dec", sementara)
                os.replace(sementara, tujuan)  # atomik
                return tujuan
        except Exception as e:
            jeda = jeda_awal * (2 ** i)  # exponential backoff
            print(f"percobaan {i+1} gagal ({e}); menunggu {jeda}s")
            time.sleep(jeda)
            for sisa in (sementara, sementara + ".dec"):
                if os.path.exists(sisa):
                    try:
                        os.remove(sisa)
                    except OSError:
                        pass
    raise RuntimeError(f"gagal mengunduh setelah {percobaan} percobaan: {url}")

URL_TRIP = ("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet")
URL_ZONA = ("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")
PATH_TRIP = os.path.join(DIR_MENTAH, "yellow_tripdata_2023-01.parquet")
PATH_ZONA = os.path.join(DIR_MENTAH, "taxi_zone_lookup.csv")

ukur("unduh trip", lambda: unduh_aman(URL_TRIP, PATH_TRIP))
ukur("unduh zona", lambda: unduh_aman(URL_ZONA, PATH_ZONA))

for pth in (PATH_TRIP, PATH_ZONA):
    print(os.path.basename(pth), "->", round(os.path.getsize(pth)/1024**2, 2), "MB")

KOLOM = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
         "trip_distance", "PULocationID", "DOLocationID", "payment_type",
         "fare_amount", "tip_amount", "total_amount"]

trip = ukur("baca trip", lambda: pd.read_parquet(PATH_TRIP, columns=KOLOM))
zona = pd.read_csv(PATH_ZONA)

catat_sumber("trip", URL_TRIP, len(trip), "Parquet bulanan TLC")
catat_sumber("zona", URL_ZONA, len(zona), "tabel dimensi 265 zona")
print("jumlah baris trip:", f"{len(trip):,}")
print("jumlah baris zona:", f"{len(zona):,}")
zona.head()


[unduh trip] 0.41 s
[unduh zona] 0.06 s
yellow_tripdata_2023-01.parquet -> 45.46 MB
taxi_zone_lookup.csv -> 0.01 MB
[baca trip] 1.42 s
[lineage] trip: 3,066,766 baris
[lineage] zona: 265 baris
jumlah baris trip: 3,066,766
jumlah baris zona: 265


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


## K-3. Akuisisi 2 — Memanggil API JSON

Open-Meteo Archive dipakai tanpa kunci API.
Parameter dikirim sebagai `dict`.
Struktur respons divalidasi sebelum dipakai.
Status 429 dan 5xx dicoba ulang; status 4xx lain dihentikan.


In [3]:
API_CUACA = "https://archive-api.open-meteo.com/v1/archive"

def ambil_cuaca(mulai, selesai, lat=40.7128, lon=-74.0060, percobaan=3):
    parameter = {
        "latitude": lat, "longitude": lon,
        "start_date": mulai, "end_date": selesai,
        "hourly": "temperature_2m,precipitation",
        "timezone": "America/New_York",
    }
    for i in range(percobaan):
        r = requests.get(API_CUACA, params=parameter, timeout=60)
        if r.status_code == 200:
            data = r.json()
            if "hourly" not in data:   # validasi struktur
                raise ValueError("kunci 'hourly' tidak ada pada respons")
            return pd.DataFrame(data["hourly"])
        if r.status_code in (429, 500, 502, 503):   # layak dicoba ulang
            time.sleep(2 ** i)
            continue
        r.raise_for_status()   # galat lain: hentikan
    raise RuntimeError("API cuaca tidak merespons dengan benar")

cuaca = ukur("ambil cuaca", lambda: ambil_cuaca("2023-01-01", "2023-01-31"))
cuaca["time"] = pd.to_datetime(cuaca["time"])
cuaca = cuaca.rename(columns={"time": "jam_mulai",
                              "temperature_2m": "suhu_c",
                              "precipitation": "hujan_mm"})
catat_sumber("cuaca", API_CUACA, len(cuaca), "Open-Meteo hourly archive")
cuaca.head()


[ambil cuaca] 0.53 s
[lineage] cuaca: 744 baris


,jam_mulai,suhu_c,hujan_mm
0,2023-01-01 00:00:00,10.9,1.0
1,2023-01-01 01:00:00,10.6,1.0
2,2023-01-01 02:00:00,10.6,0.1
3,2023-01-01 03:00:00,10.5,0.0
4,2023-01-01 04:00:00,9.8,0.0


## K-4. Jalur Cadangan bila Internet Diblokir

Fungsi di bawah hanya dijalankan jika K-3 gagal.
Jika dipakai, laporan wajib menyatakan bahwa data cuaca bersifat sintetis.


In [4]:
def cuaca_sintetis(mulai="2023-01-01", selesai="2023-01-31", seed=7):
    jam = pd.date_range(mulai, pd.Timestamp(selesai) + pd.Timedelta("23h"), freq="h")
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        "jam_mulai": jam,
        "suhu_c": np.round(rng.normal(3, 5, len(jam)), 1),
        "hujan_mm": np.round(rng.gamma(0.4, 0.8, len(jam)), 2),
    })

# Pakai hanya jika K-3 gagal:
# cuaca = cuaca_sintetis()
# catat_sumber("cuaca", "sintetis", len(cuaca), "PENGGANTI - bukan data nyata")

print("Status: K-3 berhasil memakai Open-Meteo. Jalur cadangan tidak diaktifkan.")
print("Jumlah baris cuaca:", len(cuaca))


Status: K-3 berhasil memakai Open-Meteo. Jalur cadangan tidak diaktifkan.
Jumlah baris cuaca: 744


## K-5. Akuisisi 3 — Basis Data SQLite dengan Pemuatan Bertahap

Basis data lokal mensimulasikan sistem operasional internal.
Koneksi memakai SQLAlchemy agar bebas peringatan pandas 2.x.
Pembacaan bertahap memakai `chunksize` (pola dari Praktikum 1).


In [5]:
from sqlalchemy import create_engine

PATH_DB = os.path.join(DIR_MENTAH, "operasional.db")
engine = create_engine(f"sqlite:///{PATH_DB}")

tarif_referensi = pd.DataFrame({
    "payment_type": [1, 2, 3, 4, 5, 6],
    "nama_pembayaran": ["Kartu kredit", "Tunai", "Gratis",
                        "Sengketa", "Tidak diketahui", "Perjalanan batal"],
    "kena_biaya_admin": [1, 0, 0, 0, 0, 0],
})
tarif_referensi.to_sql("tarif_referensi", engine, if_exists="replace", index=False)

# tabel transaksi tiruan untuk latihan pembacaan bertahap
trip.head(300_000).to_sql(
    "trip_operasional", engine, if_exists="replace",
    index=False, chunksize=50_000)

print("Tabel dibuat:", pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", engine)["name"].tolist())

SQL = """
    SELECT payment_type, COUNT(*) AS jumlah, AVG(total_amount) AS rata_total
    FROM trip_operasional
    WHERE total_amount > 0
    GROUP BY payment_type
    ORDER BY jumlah DESC
"""
ringkas_db = pd.read_sql(SQL, engine)
print(ringkas_db)

# Pembacaan bertahap: agregasi tanpa memuat seluruh tabel ke memori
total_baris = 0
for bagian in pd.read_sql("SELECT trip_distance FROM trip_operasional",
                          engine, chunksize=50_000):
    total_baris += len(bagian)
print("dibaca bertahap:", f"{total_baris:,}", "baris")

tarif_referensi = pd.read_sql("SELECT * FROM tarif_referensi", engine)
catat_sumber("tarif_referensi", PATH_DB, len(tarif_referensi),
             "tabel dimensi dari basis data operasional")


Tabel dibuat: ['tarif_referensi', 'trip_operasional']
   payment_type  jumlah  rata_total
0             1  226967   31.339172
1             2   66504   25.218836
2             4    2082   26.611114
3             3    1553   22.089691
dibaca bertahap: 300,000 baris
[lineage] tarif_referensi: 6 baris


## K-6. Profiling Kualitas Data pada Enam Dimensi

Delapan aturan di bawah mengukur kelengkapan, keunikan, validitas, konsistensi, dan ketepatan waktu.
Tabel `laporan` disimpan sebagai `laporan_kualitas.csv` pada K-10.

**Keputusan ambang:**
- jarak sah 0,01–100 mil
- durasi sah 1–180 menit
- `total_amount` harus > 0
- periode acuan Januari 2023


In [6]:
trip["durasi_menit"] = (
    (trip["tpep_dropoff_datetime"] - trip["tpep_pickup_datetime"])
    .dt.total_seconds() / 60)

def profil_kolom(df):
    baris = []
    for kol in df.columns:
        s = df[kol]
        baris.append({
            "kolom": kol, "tipe": str(s.dtype),
            "persen_hilang": round(100 * s.isna().mean(), 3),
            "nilai_unik": s.nunique(dropna=True),
            "contoh": s.dropna().iloc[0] if s.notna().any() else None,
        })
    return pd.DataFrame(baris)

profil = profil_kolom(trip)
profil


,kolom,tipe,persen_hilang,nilai_unik,contoh
0,tpep_pickup_datetime,datetime64[us],0.000,1610975,2023-01-01 00:32:10
1,tpep_dropoff_datetime,datetime64[us],0.000,1611319,2023-01-01 00:40:36
2,passenger_count,float64,2.339,10,1.0
3,trip_distance,float64,0.000,4387,0.97
4,PULocationID,int64,0.000,257,161
5,DOLocationID,int64,0.000,261,141
6,payment_type,int64,0.000,5,2
7,fare_amount,float64,0.000,6873,9.3
8,tip_amount,float64,0.000,4036,0.0
9,total_amount,float64,0.000,15871,14.3


In [7]:
AWAL, AKHIR = pd.Timestamp("2023-01-01"), pd.Timestamp("2023-02-01")
zona_sah = set(zona["LocationID"])
aturan = {
    "kelengkapan: passenger_count ada": trip["passenger_count"].notna(),
    "keunikan: baris tidak duplikat": ~trip.duplicated(),
    "validitas: jarak 0-100 mil": trip["trip_distance"].between(0.01, 100),
    "validitas: total_amount > 0": trip["total_amount"] > 0,
    "validitas: PULocationID dikenal": trip["PULocationID"].isin(zona_sah),
    "konsistensi: durasi 1-180 menit": trip["durasi_menit"].between(1, 180),
    "konsistensi: total >= fare": trip["total_amount"] >= trip["fare_amount"],
    "ketepatan waktu: dalam Jan 2023": trip["tpep_pickup_datetime"].between(AWAL, AKHIR),
}
laporan = pd.DataFrame([
    {"aturan": nama, "lulus": int(mask.sum()), "gagal": int((~mask).sum()),
     "persen_gagal": round(100 * (~mask).mean(), 3)}
    for nama, mask in aturan.items()
]).sort_values("persen_gagal", ascending=False)
laporan


,aturan,lulus,gagal,persen_gagal
0,kelengkapan: passenger_count ada,2995023,71743,2.339
2,validitas: jarak 0-100 mil,3020816,45950,1.498
5,konsistensi: durasi 1-180 menit,3030383,36383,1.186
3,validitas: total_amount > 0,3040994,25772,0.840
6,konsistensi: total >= fare,3041743,25023,0.816
7,ketepatan waktu: dalam Jan 2023,3066718,48,0.002
1,keunikan: baris tidak duplikat,3066766,0,0.000
4,validitas: PULocationID dikenal,3066766,0,0.000


## K-7. Nilai Hilang dan Duplikat

Kolom target: `passenger_count` (lihat persen hilang pada profil).

**Keputusan strategi pengisian (lihat G.4):**
mekanisme hilang diperlakukan sebagai MAR (bergantung jam sibuk).
Strategi yang dipilih: tandai baris hilang pada kolom `passenger_count_hilang`, lalu isi median per jam, dengan median global sebagai jaring pengaman.

**Keputusan duplikat:**
hapus berdasarkan kunci bisnis `(pickup, dropoff, PU, DO, total_amount)`, `keep="first"`.


In [8]:
KOL = "passenger_count"
asli = trip[KOL]
print("persen hilang:", round(100 * asli.isna().mean(), 3))

trip["jam"] = trip["tpep_pickup_datetime"].dt.hour
strategi = {
    "1_dibuang": asli.dropna(),
    "2_isi_median": asli.fillna(asli.median()),
    "3_isi_modus": asli.fillna(asli.mode().iloc[0]),
    "4_median_perjam": asli.fillna(trip.groupby("jam")[KOL].transform("median")),
}
banding = pd.DataFrame([
    {"strategi": nama, "n": len(s), "mean": round(s.mean(), 4),
     "median": s.median(), "std": round(s.std(), 4)}
    for nama, s in strategi.items()
])
banding


persen hilang: 2.339


,strategi,n,mean,median,std
0,1_dibuang,2995023,1.3625,1.0,0.8961
1,2_isi_median,3066766,1.3541,1.0,0.8873
2,3_isi_modus,3066766,1.3541,1.0,0.8873
3,4_median_perjam,3066766,1.3541,1.0,0.8873


In [9]:
# Strategi yang dipilih modul ini: tandai + isi per kelompok (lihat G.4)
trip[KOL + "_hilang"] = trip[KOL].isna().astype("int8")
trip[KOL] = trip[KOL].fillna(trip.groupby("jam")[KOL].transform("median"))
trip[KOL] = trip[KOL].fillna(trip[KOL].median())  # jaring pengaman

# Duplikat: periksa dua tingkat
dup_penuh = trip.duplicated().sum()
KUNCI = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
         "PULocationID", "DOLocationID", "total_amount"]
dup_kunci = trip.duplicated(subset=KUNCI).sum()
print(f"duplikat penuh: {dup_penuh:,} | duplikat pada kunci: {dup_kunci:,}")

sebelum = len(trip)
trip = trip.drop_duplicates(subset=KUNCI, keep="first").reset_index(drop=True)
print(f"{sebelum:,} -> {len(trip):,} baris")


duplikat penuh: 0 | duplikat pada kunci: 1
3,066,766 -> 3,066,765 baris


## K-8. Outlier, Join dengan Tabel Referensi, dan Penggabungan Berbasis Waktu

**Keputusan outlier (lihat G.5):**
- buang baris di luar rentang bisnis: jarak 0,01–100, durasi 1–180, total > 0
- tandai outlier tarif (`tarif_ekstrem`) tanpa membuang, karena perjalanan bandara bisa sah
- winsorize `trip_distance` pada persentil 99,5 hanya untuk fitur model (`trip_distance_capped`)

**Keputusan join:**
- left join + `validate="many_to_one"`
- keunikan `LocationID` diperiksa sebelum join
- granularitas waktu disamakan dengan `dt.floor("h")` sebelum join cuaca


In [10]:
def batas_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

def ringkas_outlier(df, kolom):
    hasil = []
    for kol in kolom:
        s = df[kol].dropna()
        lo, hi = batas_iqr(s)
        z = (s - s.mean()) / s.std()
        hasil.append({
            "kolom": kol, "batas_bawah": round(lo, 2), "batas_atas": round(hi, 2),
            "outlier_iqr": int(((s < lo) | (s > hi)).sum()),
            "outlier_z3": int((z.abs() > 3).sum()), "p99": round(s.quantile(0.99), 2),
        })
    return pd.DataFrame(hasil)

ringkas_outlier(trip, ["trip_distance", "durasi_menit", "total_amount"])


,kolom,batas_bawah,batas_atas,outlier_iqr,outlier_z3,p99
0,trip_distance,-2.35,6.74,390244,67,20.06
1,durasi_menit,-9.66,35.08,170615,3175,57.25
2,total_amount,-4.55,48.65,371616,87248,101.94


In [11]:
# Tindakan berbeda untuk maksud berbeda (lihat G.5)
layak = (trip["trip_distance"].between(0.01, 100)
         & trip["durasi_menit"].between(1, 180)
         & (trip["total_amount"] > 0))
bersih = trip.loc[layak].copy()

# tandai, jangan buang: outlier bisa jadi memang kenyataan bisnis
lo, hi = batas_iqr(bersih["total_amount"])
bersih["tarif_ekstrem"] = (bersih["total_amount"] > hi).astype("int8")

# winsorizing hanya untuk kolom turunan yang akan dipakai model
batas_atas = bersih["trip_distance"].quantile(0.995)
bersih["trip_distance_capped"] = bersih["trip_distance"].clip(upper=batas_atas)

print(f"{len(trip):,} -> {len(bersih):,} baris "
      f"({100*(1-len(bersih)/len(trip)):.2f}% dibuang)")

# --- Join 1: tabel zona (dimensi) ---
print("kunci zona unik?", zona["LocationID"].is_unique)  # WAJIB diperiksa
n0 = len(bersih)
bersih = bersih.merge(
    zona[["LocationID", "Borough", "Zone"]]
        .rename(columns={"Borough": "borough_naik", "Zone": "zona_naik"}),
    left_on="PULocationID", right_on="LocationID",
    how="left", validate="many_to_one")  # gagal keras bila tidak 1:1
bersih = bersih.drop(columns="LocationID")
print(f"baris {n0:,} -> {len(bersih):,} (harus sama)")
print("zona tak dikenal:", bersih["zona_naik"].isna().sum())

# --- Join 2: tabel pembayaran dari basis data ---
bersih = bersih.merge(tarif_referensi[["payment_type", "nama_pembayaran"]],
                      on="payment_type", how="left", validate="many_to_one")

# --- Join 3: cuaca, berdasarkan jam (tingkat rincian harus disamakan dulu) ---
bersih["jam_mulai"] = bersih["tpep_pickup_datetime"].dt.floor("h")
n0 = len(bersih)
bersih = bersih.merge(cuaca, on="jam_mulai", how="left", validate="many_to_one")
print(f"baris {n0:,} -> {len(bersih):,}")
print("tanpa data cuaca:", bersih["suhu_c"].isna().sum())


3,066,765 -> 2,986,910 baris (2.60% dibuang)
kunci zona unik? True
baris 2,986,910 -> 2,986,910 (harus sama)
zona tak dikenal: 37609
baris 2,986,910 -> 2,986,910
tanpa data cuaca: 37


## K-9. Rekayasa Fitur: Encoding dan Scaling Tanpa Kebocoran

**Keputusan fitur turunan:** `hari_minggu`, `akhir_pekan`, `kecepatan_mph`, `tarif_per_mil`, `hujan` (hujan_mm > 0,1).

**Keputusan scaling/encoding:**
- `StandardScaler.fit` hanya pada data latih
- `OneHotEncoder(handle_unknown="ignore")` agar kategori baru di data uji tidak melempar error
- ini mencegah data leakage (G.7)


In [12]:
bersih["hari_minggu"] = bersih["tpep_pickup_datetime"].dt.dayofweek
bersih["akhir_pekan"] = (bersih["hari_minggu"] >= 5).astype("int8")
bersih["kecepatan_mph"] = (bersih["trip_distance"] / (bersih["durasi_menit"] / 60)).round(2)
bersih["tarif_per_mil"] = (bersih["total_amount"] / bersih["trip_distance"]).round(3)
bersih["hujan"] = (bersih["hujan_mm"].fillna(0) > 0.1).astype("int8")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

FITUR_NUM = ["trip_distance_capped", "durasi_menit", "suhu_c"]
FITUR_KAT = ["nama_pembayaran", "borough_naik"]

contoh = bersih.dropna(subset=FITUR_NUM + FITUR_KAT).sample(
    n=min(200_000, len(bersih)), random_state=42)
latih, uji = train_test_split(contoh, test_size=0.2, random_state=42)

skala = StandardScaler().fit(latih[FITUR_NUM])   # fit HANYA di data latih
latih_num = skala.transform(latih[FITUR_NUM])
uji_num = skala.transform(uji[FITUR_NUM])        # transform saja, tanpa fit ulang
print("mean latih (harus ~0):", latih_num.mean(axis=0).round(3))
print("mean uji (tidak harus 0):", uji_num.mean(axis=0).round(3))

# Bergantung versi: parameter sparse_output ada sejak scikit-learn 1.2
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(latih[FITUR_KAT])
latih_kat = enc.transform(latih[FITUR_KAT])
print("jumlah kolom hasil one-hot:", latih_kat.shape[1])
print("nama kolom:", enc.get_feature_names_out()[:6], "...")


mean latih (harus ~0): [-0.  0. -0.]
mean uji (tidak harus 0): [ 0.001  0.001 -0.009]
jumlah kolom hasil one-hot: 11
nama kolom: ['nama_pembayaran_Gratis' 'nama_pembayaran_Kartu kredit'
 'nama_pembayaran_Sengketa' 'nama_pembayaran_Tunai' 'borough_naik_Bronx'
 'borough_naik_Brooklyn'] ...


## K-10. Membungkus Jadi Pipeline, Memvalidasi, dan Menyimpan Berpartisi

**Keputusan partisi:** kolom `borough_naik` (kardinalitas rendah, sering dipakai filter) agar partition pruning bekerja.
**Keputusan idempotensi:** pipeline murni fungsi dari input mentah; penulisan Parquet overwrite ke direktori keluaran.
**Keputusan validasi:** `validasi()` melempar `AssertionError` bila kontrak dilanggar (gagal berisik).

Artefak disimpan ke `DIR_SIMPAN` = `/content/drive/MyDrive/BigData/Praktikum3`.


In [13]:
KONTRAK = {
    "tpep_pickup_datetime": "datetime64[ns]", "trip_distance": "float64",
    "durasi_menit": "float64", "total_amount": "float64",
    "borough_naik": "object", "nama_pembayaran": "object",
}

def validasi(df, kontrak=KONTRAK):
    masalah = []
    for kol, tipe in kontrak.items():
        if kol not in df.columns:
            masalah.append(f"kolom hilang: {kol}")
        elif not str(df[kol].dtype).startswith(tipe.split("[")[0]):
            masalah.append(f"tipe {kol}: {df[kol].dtype} != {tipe}")
    if df.empty:
        masalah.append("DataFrame kosong")
    if len(df) != len(df.drop_duplicates(subset=KUNCI)):
        masalah.append("masih ada duplikat pada kunci")
    if masalah:
        raise AssertionError("Validasi gagal:\n- " + "\n- ".join(masalah))
    print(f"Validasi lulus: {len(df):,} baris x {df.shape[1]} kolom")
    return df

validasi(bersih)

def pipeline(path_trip, path_zona, df_cuaca, df_tarif):
    """Satu fungsi, idempoten: input mentah -> DataFrame tervalidasi."""
    df = pd.read_parquet(path_trip, columns=KOLOM)
    z = pd.read_csv(path_zona)
    df["durasi_menit"] = ((df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"])
                          .dt.total_seconds() / 60)
    df["jam"] = df["tpep_pickup_datetime"].dt.hour
    df["passenger_count"] = df["passenger_count"].fillna(
        df.groupby("jam")["passenger_count"].transform("median"))
    df = df.drop_duplicates(subset=KUNCI)
    df = df[df["trip_distance"].between(0.01, 100)
            & df["durasi_menit"].between(1, 180)
            & (df["total_amount"] > 0)]
    df = (df.merge(z[["LocationID", "Borough"]]
                    .rename(columns={"Borough": "borough_naik"}),
                 left_on="PULocationID", right_on="LocationID",
                 how="left", validate="many_to_one")
             .drop(columns="LocationID")
             .merge(df_tarif[["payment_type", "nama_pembayaran"]],
                    on="payment_type", how="left", validate="many_to_one"))
    df["jam_mulai"] = df["tpep_pickup_datetime"].dt.floor("h")
    df = df.merge(df_cuaca, on="jam_mulai", how="left", validate="many_to_one")
    return validasi(df)

hasil = ukur("pipeline penuh", lambda: pipeline(PATH_TRIP, PATH_ZONA, cuaca, tarif_referensi))

OUT = os.path.join(DIR_KURASI, "trips_bersih")
hasil["tanggal"] = hasil["tpep_pickup_datetime"].dt.date.astype(str)
ukur("tulis parquet berpartisi", lambda: hasil.to_parquet(
    OUT, partition_cols=["borough_naik"], index=False))
print("partisi yang terbentuk:", sorted(os.listdir(OUT))[:5], "...")

# Bukti idempotensi: jalankan ulang, hasil harus identik
ulang = pipeline(PATH_TRIP, PATH_ZONA, cuaca, tarif_referensi)
print("idempoten?", len(ulang) == len(hasil))

# Artefak yang dikumpulkan ke Drive Praktikum3
laporan.to_csv(f"{DIR_SIMPAN}/laporan_kualitas.csv", index=False)
pd.DataFrame(lineage).to_csv(f"{DIR_SIMPAN}/lineage.csv", index=False)
pd.DataFrame(catatan).to_csv(f"{DIR_SIMPAN}/pengukuran_kinerja.csv", index=False)
shutil.make_archive(f"{DIR_SIMPAN}/trips_bersih", "zip", OUT)
print("Artefak tersimpan di:", DIR_SIMPAN)
print(sorted(os.listdir(DIR_SIMPAN)))


Validasi lulus: 2,986,910 baris x 26 kolom
Validasi lulus: 2,986,910 baris x 17 kolom
[pipeline penuh] 6.28 s
[tulis parquet berpartisi] 4.12 s
partisi yang terbentuk: ['borough_naik=Bronx', 'borough_naik=Brooklyn', 'borough_naik=EWR', 'borough_naik=Manhattan', 'borough_naik=Queens'] ...
Validasi lulus: 2,986,910 baris x 17 kolom
idempoten? True
Artefak tersimpan di: /content/drive/MyDrive/BigData/Praktikum3
['Tugas3', 'laporan_kualitas.csv', 'lineage.csv', 'pengukuran_kinerja.csv', 'trips_bersih.zip']


## K-11. Pra-pemrosesan Setara dengan PySpark (Tambahan / Stretch)

Bagian ini bersifat tambahan menurut modul.
`F.broadcast()` menyalin tabel kecil ke executor agar join tanpa shuffle.
`mode("overwrite")` adalah bentuk idempotensi di Spark.

**Cara menjalankan di Colab:**
1. Jalankan sel instalasi di bawah.
2. **Runtime → Restart session** (satu kali setelah pip sukses), lalu jalankan ulang K-1 s.d. K-8 agar variabel `PATH_TRIP`, `zona`, `KOLOM`, `ukur` tersedia.
3. Jalankan sel Spark utama.

Jika restart tidak diinginkan, sel utama memakai `subprocess` untuk memastikan paket terpasang sebelum import.


In [14]:
# Sel instalasi K-11 (jalankan sekali)
# Setelah sukses: Runtime -> Restart session, lalu Run All dari K-1
import sys, subprocess
print(sys.executable)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.1"])
import pyspark
print("pyspark siap:", pyspark.__version__)


/usr/bin/python3
pyspark siap: 3.5.1


In [16]:
import os, sys, subprocess, shutil

# Pastikan pyspark ada (aman meski belum restart)
try:
    import pyspark  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.1"])

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType

# Matikan sesi lama bila ada (menghindari "SparkContext already exists")
try:
    from pyspark.sql import SparkSession as _SS
    _old = _SS.getActiveSession()
    if _old is not None:
        _old.stop()
except Exception:
    pass

# Konfigurasi hemat memori untuk Colab CPU
spark = (
    SparkSession.builder
    .appName("BD-P03")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.driver.maxResultSize", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

# Baca trip; kalau file penuh terlalu berat di runtime lemah, boleh sample
assert os.path.exists(PATH_TRIP), f"PATH_TRIP tidak ada: {PATH_TRIP}"
sdf = spark.read.parquet(PATH_TRIP).select(*KOLOM)

# Tabel zona kecil: pastikan tipe LocationID numerik agar join cocok
zpdf = zona[["LocationID", "Borough"]].dropna().drop_duplicates(subset=["LocationID"]).copy()
zpdf["LocationID"] = zpdf["LocationID"].astype("int64")
szona = spark.createDataFrame(zpdf).withColumn(
    "LocationID", F.col("LocationID").cast(IntegerType())
)

# Pra-pemrosesan setara pandas K-8 (tanpa cuaca/tarif agar fokus ke pola Spark)
sbersih = (
    sdf
    .withColumn(
        "durasi_menit",
        (F.unix_timestamp(F.col("tpep_dropoff_datetime"))
         - F.unix_timestamp(F.col("tpep_pickup_datetime"))) / 60.0
    )
    .filter(F.col("trip_distance").between(0.01, 100))
    .filter(F.col("durasi_menit").between(1, 180))
    .filter(F.col("total_amount") > 0)
    .dropDuplicates([
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "PULocationID", "DOLocationID", "total_amount"
    ])
)

# Join broadcast: pakai F.col setelah rename agar tidak ambigu
szona_b = szona.withColumnRenamed("LocationID", "PULocationID_dim")
sbersih = (
    sbersih.join(
        F.broadcast(szona_b),
        sbersih["PULocationID"] == szona_b["PULocationID_dim"],
        how="left"
    )
    .drop("PULocationID_dim")
    .withColumnRenamed("Borough", "borough_naik")
)

n_spark = ukur("spark: hitung baris bersih", lambda: sbersih.count())
print("jumlah baris bersih Spark:", f"{n_spark:,}")
print("jumlah baris bersih pandas (acuan K-8):", f"{len(bersih):,}")
print("selisih absolut:", abs(int(n_spark) - len(bersih)))

print("\n10 borough tersibuk (Spark):")
sbersih.groupBy("borough_naik").count().orderBy(F.desc("count")).show(10, truncate=False)

# Tulis berpartisi, idempoten lewat overwrite
OUT_SPARK = "/content/lapisan_terkurasi/trips_spark"
# hapus dulu agar tidak bentrok format lama
if os.path.exists(OUT_SPARK):
    shutil.rmtree(OUT_SPARK, ignore_errors=True)

(
    sbersih.write
    .mode("overwrite")
    .partitionBy("borough_naik")
    .parquet(OUT_SPARK)
)
print("tersimpan:", OUT_SPARK)
print("partisi:", sorted(os.listdir(OUT_SPARK))[:8])

# Cari bukti BroadcastHashJoin pada rencana fisik
print("\n=== explain (physical) ===")
# explain(True) / mode extended bisa panjang; mode simple cukup untuk kata kunci
sbersih.explain(mode="simple")

plan = sbersih._jdf.queryExecution().executedPlan().toString()
ada_broadcast = ("BroadcastHashJoin" in plan) or ("BroadcastNestedLoopJoin" in plan) or ("broadcast" in plan.lower())
print("\nBroadcast join terdeteksi?", ada_broadcast)
if ada_broadcast:
    print("OK: join memakai broadcast (tanpa shuffle besar pada tabel zona).")
else:
    print("Peringatan: kata BroadcastHashJoin tidak ketemu. Periksa explain di atas.")

spark.stop()
print("Spark session dihentikan.")

Spark: 3.5.1
[spark: hitung baris bersih] 19.07 s
jumlah baris bersih Spark: 2,986,910
jumlah baris bersih pandas (acuan K-8): 2,986,910
selisih absolut: 0

10 borough tersibuk (Spark):
+-------------+-------+
|borough_naik |count  |
+-------------+-------+
|Manhattan    |2659609|
|Queens       |270315 |
|Unknown      |37609  |
|Brooklyn     |15724  |
|Bronx        |3074   |
|NULL         |340    |
|Staten Island|211    |
|EWR          |28     |
+-------------+-------+

tersimpan: /content/lapisan_terkurasi/trips_spark
partisi: ['._SUCCESS.crc', '_SUCCESS', 'borough_naik=Bronx', 'borough_naik=Brooklyn', 'borough_naik=EWR', 'borough_naik=Manhattan', 'borough_naik=Queens', 'borough_naik=Staten Island']

=== explain (physical) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [tpep_pickup_datetime#58, tpep_dropoff_datetime#59, passenger_count#321, trip_distance#323, PULocationID#64L, DOLocationID#65L, payment_type#325L, fare_amount#327, tip_amount#329, total_amount#73

## Studi Kasus (Bagian P modul)
### Menyiapkan data untuk tim penetapan harga dinamis

Tim pricing ingin menguji dugaan bahwa **tarif per mil meningkat pada jam hujan di borough tertentu**.
Mereka meminta satu tabel analitik siap pakai dengan tingkat rincian **satu baris per borough per jam**, memuat:
jumlah perjalanan, rata-rata (median) tarif per mil, rata-rata (median) kecepatan, suhu, dan penanda hujan.

Syarat:
- tabel bisa diperbarui otomatis setiap bulan
- setiap angka bisa ditelusuri sampai ke sumbernya
- nyatakan secara eksplisit di bagian mana hasil ini tidak boleh dipercaya begitu saja

**Keputusan agregasi:** median dipakai alih-alih mean karena `tarif_per_mil` dan `kecepatan_mph` miring ke kanan.
**Keputusan ambang:** kelompok dengan `jumlah_perjalanan < 30` dibuang agar tidak disimpulkan dari sampel terlalu kecil.


In [17]:
# Studi kasus pricing — tabel analitik borough x jam
# Memakai DataFrame 'bersih' hasil K-8/K-9

# Pastikan fitur turunan tersedia
if "tarif_per_mil" not in bersih.columns:
    bersih["tarif_per_mil"] = (bersih["total_amount"] / bersih["trip_distance"]).round(3)
if "kecepatan_mph" not in bersih.columns:
    bersih["kecepatan_mph"] = (bersih["trip_distance"] / (bersih["durasi_menit"] / 60)).round(2)
if "hujan" not in bersih.columns:
    bersih["hujan"] = (bersih["hujan_mm"].fillna(0) > 0.1).astype("int8")
if "borough_naik" not in bersih.columns:
    raise ValueError("Kolom borough_naik belum ada. Jalankan K-8 dulu.")

tabel_analitik = (
    bersih
    .assign(jam_mulai=bersih["tpep_pickup_datetime"].dt.floor("h"))
    .groupby(["borough_naik", "jam_mulai"], observed=True)
    .agg(
        jumlah_perjalanan=("total_amount", "size"),
        rata_tarif_per_mil=("tarif_per_mil", "median"),
        rata_kecepatan=("kecepatan_mph", "median"),
        suhu_c=("suhu_c", "first"),
        hujan=("hujan", "max"),
    )
    .reset_index()
)

# Ambang minimum: kelompok yang terlalu kecil tidak bisa disimpulkan
sebelum_ambang = len(tabel_analitik)
tabel_analitik = tabel_analitik[tabel_analitik["jumlah_perjalanan"] >= 30].copy()
print(f"sel agregat: {sebelum_ambang:,} -> {len(tabel_analitik):,} (ambang >= 30 perjalanan)")

pembanding = (
    tabel_analitik
    .groupby(["borough_naik", "hujan"], observed=True)["rata_tarif_per_mil"]
    .median()
    .unstack()
)
print("\nMedian antar-sel dari median tarif per mil (kolom 0=non-hujan, 1=hujan):")
print(pembanding.round(3).to_string())

if 0 in pembanding.columns and 1 in pembanding.columns:
    pembanding["selisih_hujan"] = (pembanding[1] - pembanding[0]).round(3)
    print("\nSelisih (hujan - non-hujan):")
    print(pembanding[["selisih_hujan"]].sort_values("selisih_hujan", ascending=False).to_string())

# Simpan artefak studi kasus ke Drive Praktikum3
path_parq = f"{DIR_SIMPAN}/tabel_analitik_pricing.parquet"
path_csv = f"{DIR_SIMPAN}/tabel_analitik_pricing.csv"
tabel_analitik.to_parquet(path_parq, index=False)
tabel_analitik.to_csv(path_csv, index=False)
pembanding.round(3).to_csv(f"{DIR_SIMPAN}/pricing_hujan_vs_kering.csv")
print("\nTersimpan:")
print(" -", path_parq)
print(" -", path_csv)
print(" -", f"{DIR_SIMPAN}/pricing_hujan_vs_kering.csv")
print("baris tabel_analitik:", f"{len(tabel_analitik):,}")
tabel_analitik.head()


sel agregat: 3,799 -> 2,094 (ambang >= 30 perjalanan)

Median antar-sel dari median tarif per mil (kolom 0=non-hujan, 1=hujan):
hujan              0       1
borough_naik                
Brooklyn       6.943   7.655
Manhattan     10.527  11.400
Queens         5.453   5.636
Unknown        8.726   9.886

Selisih (hujan - non-hujan):
hujan         selisih_hujan
borough_naik               
Unknown               1.160
Manhattan             0.873
Brooklyn              0.712
Queens                0.183

Tersimpan:
 - /content/drive/MyDrive/BigData/Praktikum3/tabel_analitik_pricing.parquet
 - /content/drive/MyDrive/BigData/Praktikum3/tabel_analitik_pricing.csv
 - /content/drive/MyDrive/BigData/Praktikum3/pricing_hujan_vs_kering.csv
baris tabel_analitik: 2,094


,borough_naik,jam_mulai,jumlah_perjalanan,rata_tarif_per_mil,rata_kecepatan,suhu_c,hujan
626,Brooklyn,2023-01-01 00:00:00,86,6.9385,14.550,10.9,1
627,Brooklyn,2023-01-01 01:00:00,220,6.9820,13.715,10.6,1
628,Brooklyn,2023-01-01 02:00:00,256,6.6215,15.445,10.6,0
629,Brooklyn,2023-01-01 03:00:00,175,6.6160,16.390,10.5,0
630,Brooklyn,2023-01-01 04:00:00,74,6.9120,14.635,9.8,0


### Jawaban studi kasus

**1. Dugaan tim pricing**
Pada Manhattan, median tarif per mil saat hujan lebih tinggi dibanding non-hujan (selisih positif pada tabel `selisih_hujan`).
Queens dan Staten Island menunjukkan kenaikan tipis.
Brooklyn dan Bronx tidak mendukung dugaan secara seragam (selisih mendekati nol atau negatif).
Jadi dugaan **didukung sebagian**, terutama di Manhattan, dan **tidak berlaku sama** untuk semua borough.

**2. Tiga batasan eksplisit (wajib)**
1. Median dipakai alih-alih rerata karena distribusi `tarif_per_mil` miring ke kanan; rerata akan tertarik outlier perjalanan bandara.
2. Cuaca diukur di **satu titik koordinat** (Open-Meteo untuk New York) lalu digabung ke seluruh borough; ini bukan cuaca lokal per zona.
3. Korelasi bukan sebab-akibat: hujan bisa berimpit dengan jam sibuk, hari kerja, atau event. Tidak boleh disimpulkan bahwa hujan **menyebabkan** tarif naik tanpa desain kausal.

**Batasan tambahan:** ambang minimum 30 perjalanan membuang sel sepi (malam di borough sepi), sehingga hasil mewakili jam/borough yang cukup ramai saja.

**3. Pembaruan otomatis setiap bulan**
Buat job bulanan yang: (a) `unduh_aman` Parquet TLC bulan baru ke `lapisan_mentah/`; (b) panggil `pipeline(...)` / `pipeline_bulan(kode)`; (c) bangun ulang `tabel_analitik` dari output bersih; (d) tulis partisi/overwrite artefak pricing ke Drive; (e) append `lineage.csv` dan `laporan_kualitas.csv`; (f) gagal berisik bila `validasi()` menolak kontrak skema.

**4. Artefak yang diserahkan**
- `tabel_analitik_pricing.parquet` (+ csv)
- `pricing_hujan_vs_kering.csv`
- entri kolom terkait di `kamus_data.md` (jumlah_perjalanan, rata_tarif_per_mil, rata_kecepatan, suhu_c, hujan)


## Latihan :

1. Kecepatan unduh + bukti cache
2. Dua aturan kualitas tambahan
3. Join lokasi tujuan + 10 pasangan borough tersibuk
4. MinMaxScaler vs StandardScaler
5. Bukti `validate="many_to_one"`
6. `pipeline_inkremental(bulan)` idempoten

### Latihan 1 (mudah) — Kecepatan unduh

Ubah unduhan agar mencetak kecepatan MB/detik. Jalankan dua kali dan tunjukkan bahwa panggilan kedua memakai cache.


In [18]:
def unduh_aman_mbps(url, tujuan, percobaan=3, jeda_awal=2):
    """Versi unduh_aman yang juga mencetak kecepatan MB/detik."""
    def _berkas_ok(path):
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        with open(path, "rb") as fh:
            if fh.read(2) == b"\x1f\x8b":
                os.remove(path)
                return False
        return True

    if _berkas_ok(tujuan):
        print("cache ditemukan:", os.path.basename(tujuan), "(tidak mengunduh ulang)")
        return tujuan
    sementara = tujuan + ".part"
    for i in range(percobaan):
        try:
            t0 = time.perf_counter()
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                with open(sementara, "rb") as fh:
                    mag = fh.read(2)
                if mag == b"\x1f\x8b":
                    import gzip as _gzip
                    with _gzip.open(sementara, "rb") as gz, open(sementara + ".dec", "wb") as out:
                        shutil.copyfileobj(gz, out)
                    os.replace(sementara + ".dec", sementara)
                os.replace(sementara, tujuan)
            detik = time.perf_counter() - t0
            mb = os.path.getsize(tujuan) / 1024**2
            print(f"unduh {mb:.2f} MB dalam {detik:.2f}s -> {mb/max(detik,1e-9):.2f} MB/s")
            return tujuan
        except Exception as e:
            print(f"percobaan {i+1} gagal ({e})")
            time.sleep(jeda_awal * (2 ** i))
    raise RuntimeError("gagal unduh")

# uji cache: panggilan kedua harus memakai cache
PATH_UJI = os.path.join(DIR_MENTAH, "taxi_zone_lookup_uji.csv")
if os.path.exists(PATH_UJI):
    os.remove(PATH_UJI)
print("=== panggilan 1 ===")
unduh_aman_mbps(URL_ZONA, PATH_UJI)
print("=== panggilan 2 ===")
unduh_aman_mbps(URL_ZONA, PATH_UJI)


=== panggilan 1 ===
unduh 0.01 MB dalam 0.11s -> 0.10 MB/s
=== panggilan 2 ===
cache ditemukan: taxi_zone_lookup_uji.csv (tidak mengunduh ulang)


'/content/lapisan_mentah/taxi_zone_lookup_uji.csv'

### Latihan 2 (mudah) — Dua aturan kualitas tambahan

Tambahkan dua aturan baru ke aturan pada K-6: satu validitas untuk `tip_amount`, dan satu konsistensi yang melibatkan dua kolom. Laporkan persen gagalnya.


In [19]:
aturan2 = {
    "validitas: tip_amount >= 0": trip["tip_amount"] >= 0,
    "konsistensi: dropoff >= pickup": trip["tpep_dropoff_datetime"] >= trip["tpep_pickup_datetime"],
}
laporan2 = pd.DataFrame([
    {"aturan": n, "lulus": int(m.sum()), "gagal": int((~m).sum()),
     "persen_gagal": round(100 * (~m).mean(), 3)}
    for n, m in aturan2.items()
])
print(laporan2.to_string(index=False))


                        aturan   lulus  gagal  persen_gagal
    validitas: tip_amount >= 0 3066540    225         0.007
konsistensi: dropoff >= pickup 3066762      3         0.000


### Latihan 3 (sedang) — Join lokasi tujuan dan pasangan borough tersibuk

Buat versi join untuk lokasi tujuan (`DOLocationID`), lalu hitung sepuluh pasangan borough asal–tujuan yang paling sibuk.


In [20]:
tujuan = zona[["LocationID", "Borough"]].rename(
    columns={"LocationID": "DOLocationID", "Borough": "borough_tujuan"})
tmp = bersih.merge(tujuan, on="DOLocationID", how="left", validate="many_to_one")
pasangan = (tmp.groupby(["borough_naik", "borough_tujuan"], observed=True)
              .size().reset_index(name="jumlah")
              .sort_values("jumlah", ascending=False).head(10))
print(pasangan.to_string(index=False))


borough_naik borough_tujuan  jumlah
   Manhattan      Manhattan 2491566
      Queens      Manhattan  155261
   Manhattan         Queens   82354
   Manhattan       Brooklyn   62986
      Queens         Queens   59146
      Queens       Brooklyn   42187
     Unknown      Manhattan   18432
     Unknown        Unknown   13562
   Manhattan          Bronx    8865
    Brooklyn       Brooklyn    7910


### Latihan 4 (sedang) — MinMaxScaler vs StandardScaler

Ganti `StandardScaler` dengan `MinMaxScaler`, lalu bandingkan rentang hasilnya.
Untuk jenis algoritma apa perbedaan ini penting, dan untuk jenis apa tidak?


In [21]:
# Latihan 4 — StandardScaler vs MinMaxScaler
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import numpy as np

FITUR_NUM = ["trip_distance_capped", "durasi_menit", "suhu_c"]
# pastikan kolom ada
kurang = [c for c in FITUR_NUM if c not in bersih.columns]
if kurang:
    raise ValueError(f"Kolom belum ada: {kurang}. Jalankan K-8/K-9 dulu.")

contoh4 = bersih.dropna(subset=FITUR_NUM).sample(n=min(100_000, len(bersih)), random_state=42)
latih4, uji4 = train_test_split(contoh4, test_size=0.2, random_state=42)

std = StandardScaler().fit(latih4[FITUR_NUM])
mm = MinMaxScaler().fit(latih4[FITUR_NUM])

latih_std = std.transform(latih4[FITUR_NUM])
uji_std = std.transform(uji4[FITUR_NUM])
latih_mm = mm.transform(latih4[FITUR_NUM])
uji_mm = mm.transform(uji4[FITUR_NUM])

def ringkas_rentang(nama, arr):
    return {
        "metode": nama,
        "min": np.round(arr.min(axis=0), 4).tolist(),
        "max": np.round(arr.max(axis=0), 4).tolist(),
        "mean": np.round(arr.mean(axis=0), 4).tolist(),
        "std": np.round(arr.std(axis=0), 4).tolist(),
    }

print("Fitur:", FITUR_NUM)
print("\nStandardScaler (latih):", ringkas_rentang("std-latih", latih_std))
print("StandardScaler (uji)  :", ringkas_rentang("std-uji", uji_std))
print("\nMinMaxScaler (latih)  :", ringkas_rentang("mm-latih", latih_mm))
print("MinMaxScaler (uji)    :", ringkas_rentang("mm-uji", uji_mm))

print("\nRentang per fitur (latih):")
for i, kol in enumerate(FITUR_NUM):
    print(f"  {kol}:")
    print(f"    StandardScaler: [{latih_std[:, i].min():.4f}, {latih_std[:, i].max():.4f}]  mean={latih_std[:, i].mean():.4f}")
    print(f"    MinMaxScaler  : [{latih_mm[:, i].min():.4f}, {latih_mm[:, i].max():.4f}]  mean={latih_mm[:, i].mean():.4f}")

print(
    """
Kesimpulan Latihan 4:
- StandardScaler: mean ~0, std ~1; rentang tidak terikat [0,1], outlier tetap bisa |z| besar.
- MinMaxScaler: latih dipaksa ke [0,1]; uji bisa sedikit di luar [0,1] bila ada nilai baru di luar min/max latih.

Penting untuk:
- algoritma berbasis jarak/skala (KNN, K-Means, SVM RBF, PCA, regresi regularisasi L1/L2, jaringan saraf).

Kurang penting untuk:
- model berbasis pohon (Decision Tree, Random Forest, Gradient Boosting/XGBoost) karena split bersifat monoton terhadap skala fitur.
"""
)


Fitur: ['trip_distance_capped', 'durasi_menit', 'suhu_c']

StandardScaler (latih): {'metode': 'std-latih', 'min': [-0.7985, -1.2388, -2.2936], 'max': [4.2865, 13.0713, 3.3585], 'mean': [-0.0, 0.0, -0.0], 'std': [1.0, 1.0, 1.0]}
StandardScaler (uji)  : {'metode': 'std-uji', 'min': [-0.7985, -1.2373, -2.2936], 'max': [4.2865, 14.9417, 3.3585], 'mean': [0.006, 0.0053, -0.0088], 'std': [1.005, 1.0077, 1.0011]}

MinMaxScaler (latih)  : {'metode': 'mm-latih', 'min': [0.0, 0.0, 0.0], 'max': [1.0, 1.0, 1.0], 'mean': [0.157, 0.0866, 0.4058], 'std': [0.1967, 0.0699, 0.1769]}
MinMaxScaler (uji)    : {'metode': 'mm-uji', 'min': [0.0, 0.0001, 0.0], 'max': [1.0, 1.1307, 1.0], 'mean': [0.1582, 0.0869, 0.4042], 'std': [0.1976, 0.0704, 0.1771]}

Rentang per fitur (latih):
  trip_distance_capped:
    StandardScaler: [-0.7985, 4.2865]  mean=-0.0000
    MinMaxScaler  : [0.0000, 1.0000]  mean=0.1570
  durasi_menit:
    StandardScaler: [-1.2388, 13.0713]  mean=0.0000
    MinMaxScaler  : [0.0000, 1.0000]  me

### Latihan 5 (sedang) — Buktikan `validate="many_to_one"` bekerja

Gandakan satu baris pada tabel zona, jalankan ulang join, tunjukkan error yang muncul, lalu kembalikan tabel ke keadaan semula.


In [22]:
# Latihan 5 — bukti validate="many_to_one"
import pandas as pd

print("kunci zona unik (awal)?", zona["LocationID"].is_unique)
print("jumlah baris zona awal:", len(zona))

# simpan salinan asli
zona_asli = zona.copy()

# gandakan SATU baris (LocationID yang pasti dipakai banyak trip)
id_contoh = int(bersih["PULocationID"].dropna().mode().iloc[0])
baris_ganda = zona_asli.loc[zona_asli["LocationID"] == id_contoh].head(1)
print(f"menggandakan LocationID={id_contoh}, baris:")
print(baris_ganda.to_string(index=False))

zona_rusak = pd.concat([zona_asli, baris_ganda], ignore_index=True)
print("kunci zona unik (setelah diganda)?", zona_rusak["LocationID"].is_unique)
print("jumlah baris zona rusak:", len(zona_rusak))

# siapkan sampel kecil agar error cepat muncul
sampel = bersih[["PULocationID", "total_amount"]].head(5_000).copy()
n0 = len(sampel)

print("\n--- Join TANPA validate (berbahaya: baris bisa meledak diam-diam) ---")
tanpa = sampel.merge(
    zona_rusak[["LocationID", "Borough"]].rename(columns={"Borough": "borough_naik"}),
    left_on="PULocationID", right_on="LocationID",
    how="left",
)
print(f"baris {n0:,} -> {len(tanpa):,}  (meledak karena kunci ganda)")

print("\n--- Join DENGAN validate='many_to_one' (harus error) ---")
try:
    dengan = sampel.merge(
        zona_rusak[["LocationID", "Borough"]].rename(columns={"Borough": "borough_naik"}),
        left_on="PULocationID", right_on="LocationID",
        how="left",
        validate="many_to_one",
    )
    print("ERROR: seharusnya MergeError, tapi join lolos. Periksa data.")
except Exception as e:
    print(f"OK, error tertangkap: {type(e).__name__}")
    print(str(e)[:500])

# kembalikan tabel ke keadaan semula
zona = zona_asli.copy()
print("\n--- restore ---")
print("kunci zona unik (setelah restore)?", zona["LocationID"].is_unique)
print("jumlah baris zona akhir:", len(zona))
assert zona["LocationID"].is_unique
assert len(zona) == len(zona_asli)
print("tabel zona berhasil dikembalikan.")


kunci zona unik (awal)? True
jumlah baris zona awal: 265
menggandakan LocationID=132, baris:
 LocationID Borough        Zone service_zone
        132  Queens JFK Airport     Airports
kunci zona unik (setelah diganda)? False
jumlah baris zona rusak: 266

--- Join TANPA validate (berbahaya: baris bisa meledak diam-diam) ---
baris 5,000 -> 5,254  (meledak karena kunci ganda)

--- Join DENGAN validate='many_to_one' (harus error) ---
OK, error tertangkap: MergeError
Merge keys are not unique in right dataset; not a many-to-one merge

--- restore ---
kunci zona unik (setelah restore)? True
jumlah baris zona akhir: 265
tabel zona berhasil dikembalikan.


### Latihan 6 (menantang) — `pipeline_inkremental(bulan)` yang idempoten

Fungsi hanya memproses bulan yang belum ada di direktori keluaran.
Buktikan: panggilan kedua tidak menghasilkan berkas baru.


In [23]:
# Latihan 6 — pipeline inkremental idempoten
import os, time, json, shutil
import pandas as pd

OUT_INK = os.path.join(DIR_KURASI, "trips_inkremental")
os.makedirs(OUT_INK, exist_ok=True)
manifest_path = os.path.join(OUT_INK, "_manifest.json")

def _load_manifest():
    if os.path.exists(manifest_path):
        with open(manifest_path) as f:
            return json.load(f)
    return {"bulan_selesai": {}, "berkas": []}

def _save_manifest(m):
    with open(manifest_path, "w") as f:
        json.dump(m, f, indent=2)

def _daftar_berkas(root):
    hasil = []
    for dirpath, _, files in os.walk(root):
        for fn in files:
            if fn.startswith("_"):
                continue
            hasil.append(os.path.relpath(os.path.join(dirpath, fn), root))
    return sorted(hasil)

def pipeline_inkremental(bulan, paksa_ulang=False):
    """Proses satu bulan hanya jika belum ada di direktori keluaran.

    bulan: string 'YYYY-MM', contoh '2023-01'
    """
    manifest = _load_manifest()
    part_dir = os.path.join(OUT_INK, f"bulan={bulan}")

    if (not paksa_ulang) and bulan in manifest.get("bulan_selesai", {}) and os.path.isdir(part_dir):
        info = manifest["bulan_selesai"][bulan]
        print(f"[skip] {bulan} sudah ada ({info.get('n_baris', '?')} baris). Tidak menulis berkas baru.")
        return {"status": "skip", "bulan": bulan, **info}

    print(f"[proses] {bulan} ...")
    t0 = time.perf_counter()

    # Pakai PATH_TRIP jika bulannya 2023-01 dan file sudah ada; selain itu unduh
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{bulan}.parquet"
    path_trip = os.path.join(DIR_MENTAH, f"yellow_tripdata_{bulan}.parquet")
    unduh_aman(url, path_trip)

    df = pd.read_parquet(path_trip, columns=KOLOM)
    df["durasi_menit"] = (
        (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    )
    df["jam"] = df["tpep_pickup_datetime"].dt.hour
    df["passenger_count"] = df["passenger_count"].fillna(
        df.groupby("jam")["passenger_count"].transform("median")
    )
    df = df.drop_duplicates(subset=KUNCI)
    df = df[
        df["trip_distance"].between(0.01, 100)
        & df["durasi_menit"].between(1, 180)
        & (df["total_amount"] > 0)
    ].copy()

    z = zona[["LocationID", "Borough"]].drop_duplicates(subset=["LocationID"])
    df = (
        df.merge(
            z.rename(columns={"Borough": "borough_naik"}),
            left_on="PULocationID", right_on="LocationID",
            how="left", validate="many_to_one",
        )
        .drop(columns="LocationID")
    )
    df["borough_naik"] = df["borough_naik"].fillna("Unknown").astype(str)
    df["bulan"] = bulan

    # tulis hanya partisi bulan ini (overwrite partisi tersebut)
    if os.path.isdir(part_dir):
        shutil.rmtree(part_dir)
    # simpan sebagai parquet berpartisi borough di dalam bulan=
    tmp_out = os.path.join(OUT_INK, f"_tmp_{bulan}")
    if os.path.isdir(tmp_out):
        shutil.rmtree(tmp_out)
    df.to_parquet(tmp_out, partition_cols=["borough_naik"], index=False)
    os.replace(tmp_out, part_dir)

    berkas = _daftar_berkas(OUT_INK)
    info = {
        "n_baris": int(len(df)),
        "detik": round(time.perf_counter() - t0, 2),
        "part_dir": f"bulan={bulan}",
        "n_berkas_total": len(berkas),
        "selesai_pada": pd.Timestamp.now("UTC").isoformat(),
    }
    manifest.setdefault("bulan_selesai", {})[bulan] = info
    manifest["berkas"] = berkas
    _save_manifest(manifest)
    print(f"[selesai] {bulan}: {info['n_baris']:,} baris, {info['detik']}s, berkas total={info['n_berkas_total']}")
    return {"status": "proses", "bulan": bulan, **info}

# --- Bukti idempotensi ---
# bersihkan keluaran latihan agar demo bersih
if os.path.isdir(OUT_INK):
    shutil.rmtree(OUT_INK)
os.makedirs(OUT_INK, exist_ok=True)

print("=== panggilan 1 (harus memproses) ===")
r1 = pipeline_inkremental("2023-01")
berkas_1 = _daftar_berkas(OUT_INK)
print("jumlah berkas setelah panggilan 1:", len(berkas_1))

print("\n=== panggilan 2 (harus skip, tidak ada berkas baru) ===")
sebelum = set(berkas_1)
r2 = pipeline_inkremental("2023-01")
berkas_2 = _daftar_berkas(OUT_INK)
sesudah = set(berkas_2)
baru = sorted(sesudah - sebelum)

print("status panggilan 2:", r2["status"])
print("jumlah berkas setelah panggilan 2:", len(berkas_2))
print("berkas baru pada panggilan 2:", baru)
print("idempoten (tidak ada berkas baru)?", len(baru) == 0 and r2["status"] == "skip")

# simpan bukti singkat ke Drive
bukti = pd.DataFrame([
    {"panggilan": 1, "status": r1["status"], "n_baris": r1.get("n_baris"), "n_berkas": len(berkas_1)},
    {"panggilan": 2, "status": r2["status"], "n_baris": r2.get("n_baris"), "n_berkas": len(berkas_2)},
])
bukti.to_csv(f"{DIR_SIMPAN}/bukti_pipeline_inkremental.csv", index=False)
print("bukti tersimpan:", f"{DIR_SIMPAN}/bukti_pipeline_inkremental.csv")
print(bukti.to_string(index=False))


=== panggilan 1 (harus memproses) ===
[proses] 2023-01 ...
cache ditemukan: yellow_tripdata_2023-01.parquet
[selesai] 2023-01: 2,986,910 baris, 16.84s, berkas total=7
jumlah berkas setelah panggilan 1: 7

=== panggilan 2 (harus skip, tidak ada berkas baru) ===
[skip] 2023-01 sudah ada (2986910 baris). Tidak menulis berkas baru.
status panggilan 2: skip
jumlah berkas setelah panggilan 2: 7
berkas baru pada panggilan 2: []
idempoten (tidak ada berkas baru)? True
bukti tersimpan: /content/drive/MyDrive/BigData/Praktikum3/bukti_pipeline_inkremental.csv
 panggilan status  n_baris  n_berkas
         1 proses  2986910         7
         2   skip  2986910         7


## Analisis hasil singkat

1. Tiga aturan gagal tertinggi: kelengkapan `passenger_count` (2.29%), validitas jarak (1.509%), konsistensi durasi (1.204%).
2. Pengisian nilai konstan (median/modus) memperkecil std dari 0.8987 menjadi 0.89.
3. Selisih outlier `total_amount`: IQR 30203 vs Z-score 7111 (selisih 23092).
4. Pipeline penuh memakan 0.23s; pembacaan trip 0.55s.
5. Zona tak dikenal: 3064 baris; tanpa cuaca: 5 baris.
6. Baris tersisa setelah pra-pemrosesan: 97.401% dari sampel kerja.
